In [ ]:
import os
import sys

try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

    colab_src_dir = "/content/drive/MyDrive/Luca/research/unsupervised-feature-research/src"
    os.chdir(colab_src_dir)
    if colab_src_dir not in sys.path:
        sys.path.insert(0, colab_src_dir)

### Setup

In [ ]:
import sys
sys.path.insert(0, '.')

import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from transformers import AutoFeatureExtractor, AutoModel, AutoModelForImageClassification
import os
import math

from models import Autoencoder, K_Sparse_AE, WTA_FC_AE, WTA_CONV_AE
from datasets import get_data_loader, get_flattened_size, get_patch_shape

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {device}")

In [ ]:
k_population   = 0.05
k_spatial      = 0.2
k_lifetime     = 0.2
epochs         = 30
batch_size     = 128
bottleneck_dim = 250
lr             = 0.001
dataset        = "cifar10_color"  # or "mnist", "mnist_patches", "cifar10_patches", "cifar10_color"
autoencoder    = "normal"

### Dataset

In [ ]:
os.makedirs('../data', exist_ok=True)

In [ ]:
data_loader = get_data_loader(dataset, batch_size=batch_size)
input_dim   = get_flattened_size(dataset)

### Autoencoders

In [ ]:
def get_autoencoder(autoencoder_type: str):
    match autoencoder_type:
        case "normal":
            return Autoencoder(
                dim=(input_dim, bottleneck_dim)
            ).to(device)

        case "sparse":
            return K_Sparse_AE(
                dim=(input_dim, bottleneck_dim),
                k_population=k_population,
                total_epochs=epochs,
                dataset_size=len(data_loader.dataset),
            ).to(device)

        case "wta_fc":
            return WTA_FC_AE(
                dim=(input_dim, bottleneck_dim), 
                k_lifetime=k_lifetime
            ).to(device)

        case "wta_conv":
            try:
                c, h, w = get_patch_shape(dataset)
            except ValueError:
                s = int(math.sqrt(input_dim))
                c, h, w = 1, s, s

            return WTA_CONV_AE(
                dim=(c, h, w, bottleneck_dim),
                k_spatial=k_spatial,
                k_population=k_lifetime,
                total_epochs=epochs,
                dataset_size=len(data_loader.dataset),
            ).to(device)

        case _:
            raise ValueError(f"Unknown autoencoder_type={autoencoder_type!r}.")

In [ ]:
autoencoder = get_autoencoder("normal")
criterion   = nn.MSELoss()
optimizer   = torch.optim.Adam(autoencoder.parameters(), lr=lr)

### Training

In [ ]:
losses            = []
model_checkpoints = []

autoencoder.train()

epochbar  = tqdm(range(epochs), desc="Epochs")
n_batches = len(data_loader)
log_every = max(1, n_batches // 4)

for epoch in epochbar:
    epoch_loss                = 0.0
    inputs_processed_in_epoch = 0

    for step, (batch_inputs, _) in enumerate(data_loader):
        batch_size_cur = batch_inputs.shape[0]
        batch_inputs   = batch_inputs.to(device)
        optimizer.zero_grad()

        if autoencoder.uses_k_population:
            x_recon   = autoencoder(batch_inputs, epoch=epoch, inputs_processed_in_epoch=inputs_processed_in_epoch)
        else:
            x_recon   = autoencoder(batch_inputs)

        inputs_processed_in_epoch += batch_size_cur

        loss   = criterion(x_recon, batch_inputs)

        loss.backward()

        optimizer.step()

        batch_loss = loss.item()

        epoch_loss += batch_loss

        if (step + 1) % log_every == 0:
            avg = epoch_loss / (step + 1)
            losses.append(avg)


        epochbar.set_postfix({
            "normal": f"{batch_loss:.4f}",
        })

    model_checkpoints.append({
        "normal": copy.deepcopy(autoencoder),
    })

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

n_batches = len(data_loader)
log_every = max(1, n_batches // 4)
logs_per_epoch = max(1, n_batches // log_every)
xs = [i / logs_per_epoch for i in range(len(normal_losses))]

axes[0, 0].plot(xs, normal_losses, "o-")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Mean Train Loss")
axes[0, 0].set_title("Autoencoder")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(xs, sparse_losses, "o-", color="orange")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Mean Train Loss")
axes[0, 1].set_title("K-Sparse AE")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(xs, wta_fc_losses, "o-", color="green")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Mean Train Loss")
axes[1, 0].set_title("WTA FC AE")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(xs, wta_conv_losses, "o-", color="purple")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Mean Train Loss")
axes[1, 1].set_title("WTA Conv AE")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Visualizations

#### 1. Dataset

In [ ]:
patches_batch, _ = next(iter(data_loader))
n_show = min(25, patches_batch.size(0))
try:
    C, H, W = get_patch_shape(dataset)
except ValueError:
    s = int(math.sqrt(input_dim))
    C, H, W = 1, s, s
patches_vis = patches_batch[:n_show].reshape(-1, C, H, W)
grid_patches = make_grid(patches_vis, nrow=5, normalize=True, scale_each=True, padding=2)
plt.figure(figsize=(8, 8))
plt.imshow(grid_patches.permute(1, 2, 0).clamp(0, 1))
plt.axis("off")
plt.title(f"Sample {dataset} patches (n={n_show})")
plt.tight_layout()
plt.show()

#### 2. Encoder weights

In [ ]:
visualize_epoch = epochs - 1

try:
    C, H, W = get_patch_shape(dataset)
except ValueError:
    s = int(math.sqrt(input_dim))
    C, H, W = 1, s, s

cp = model_checkpoints[visualize_epoch]
x_probe, _ = next(iter(data_loader))
x_probe = x_probe[:1].to(device)

for m in [cp["normal"], cp["sparse"], cp["wta_fc"], cp["wta_conv"]]:
    m.eval()

with torch.no_grad():
    cp["normal"](x_probe)
    cp["sparse"](x_probe)
    cp["wta_fc"](x_probe)
    cp["wta_conv"](x_probe)


def mask_linear_encoder_rows(W: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    m = mask.view(-1)
    return W * m.unsqueeze(1)


def mask_conv_encoder_out(W: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    m = mask.view(-1)
    return W * m.view(-1, 1, 1, 1)


m_n = cp["normal"].last_filter_mask[0].cpu()
m_s = cp["sparse"].last_filter_mask[0].cpu()
m_f = cp["wta_fc"].last_filter_mask[0].cpu()
m_c = cp["wta_conv"].last_filter_mask[0].cpu()

max_filters = 300
W_enc_normal = mask_linear_encoder_rows(cp["normal"].detached_encoder_weights.cpu(), m_n)
W_enc_sparse = mask_linear_encoder_rows(cp["sparse"].detached_encoder_weights.cpu(), m_s)
W_enc_wta_fc = mask_linear_encoder_rows(cp["wta_fc"].detached_encoder_weights.cpu(), m_f)
W_enc_conv   = mask_conv_encoder_out(cp["wta_conv"].detached_encoder_weights.cpu(), m_c)

kernels_normal = W_enc_normal[:max_filters].reshape(-1, C, H, W)
kernels_sparse = W_enc_sparse[:max_filters].reshape(-1, C, H, W)
kernels_wta_fc = W_enc_wta_fc[:max_filters].reshape(-1, C, H, W)
kernels_conv   = W_enc_conv[:max_filters]

grid_normal   = make_grid(kernels_normal, nrow=10, normalize=True, scale_each=True, padding=1)
grid_sparse   = make_grid(kernels_sparse, nrow=10, normalize=True, scale_each=True, padding=1)
grid_wta_fc   = make_grid(kernels_wta_fc, nrow=10, normalize=True, scale_each=True, padding=1)
grid_wta_conv = make_grid(kernels_conv, nrow=10, normalize=True, scale_each=True, padding=1)

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes[0, 0].imshow(grid_normal.permute(1, 2, 0).clamp(0, 1))
axes[0, 0].axis("off")
axes[0, 0].set_title("Autoencoder (masked encoder rows)")

axes[0, 1].imshow(grid_sparse.permute(1, 2, 0).clamp(0, 1))
axes[0, 1].axis("off")
axes[0, 1].set_title("K-Sparse AE (masked encoder rows)")

axes[1, 0].imshow(grid_wta_fc.permute(1, 2, 0).clamp(0, 1))
axes[1, 0].axis("off")
axes[1, 0].set_title("WTA-FC AE (masked encoder rows)")

axes[1, 1].imshow(grid_wta_conv.permute(1, 2, 0).clamp(0, 1))
axes[1, 1].axis("off")
axes[1, 1].set_title("WTA-CONV AE (masked encoder filters)")

plt.suptitle(f"Encoder weights × sparsity filter mask (epoch {visualize_epoch}, probe batch n=1)")
plt.tight_layout()
plt.show()

#### 3. Decoder weights

In [ ]:
visualize_epoch = epochs - 1

try:
    C, H, W = get_patch_shape(dataset)
except ValueError:
    s = int(math.sqrt(input_dim))
    C, H, W = 1, s, s

cp = model_checkpoints[visualize_epoch]
max_filters = 300

W_dec_normal = cp["normal"].detached_decoder_weights.cpu().T  # (bottleneck_dim, input_dim)
W_dec_sparse = cp["sparse"].detached_decoder_weights.cpu().T
W_dec_wta_fc = cp["wta_fc"].detached_decoder_weights.cpu().T
W_dec_conv   = cp["wta_conv"].detached_decoder_weights.cpu()

kernels_dec_normal = W_dec_normal[:max_filters].reshape(-1, C, H, W)
kernels_dec_sparse = W_dec_sparse[:max_filters].reshape(-1, C, H, W)
kernels_dec_wta_fc = W_dec_wta_fc[:max_filters].reshape(-1, C, H, W)
kernels_dec_conv   = W_dec_conv[:max_filters]

grid_dec_normal   = make_grid(kernels_dec_normal, nrow=10, normalize=True, scale_each=True, padding=1)
grid_dec_sparse   = make_grid(kernels_dec_sparse, nrow=10, normalize=True, scale_each=True, padding=1)
grid_dec_wta_fc   = make_grid(kernels_dec_wta_fc, nrow=10, normalize=True, scale_each=True, padding=1)
grid_dec_wta_conv = make_grid(kernels_dec_conv,   nrow=10, normalize=True, scale_each=True, padding=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0, 0].imshow(grid_dec_normal.permute(1, 2, 0).clamp(0, 1))
axes[0, 0].axis("off")
axes[0, 0].set_title("Autoencoder (decoder)")

axes[0].imshow(grid_dec_sparse.permute(1, 2, 0).clamp(0, 1))
axes[0].axis("off")
axes[0].set_title("K-Sparse AE (decoder)")

axes[1, 0].imshow(grid_dec_wta_fc.permute(1, 2, 0).clamp(0, 1))
axes[1, 0].axis("off")
axes[1, 0].set_title("WTA-FC AE (decoder)")

axes[1].imshow(grid_dec_wta_conv.permute(1, 2, 0).clamp(0, 1))
axes[1].axis("off")
axes[1].set_title("WTA-CONV AE (decoder)")

plt.suptitle(f"Decoder weights (epoch {visualize_epoch})")
plt.tight_layout()
plt.show()

#### 4. Reconstructions (25 random patches)

In [ ]:
visualize_epoch = epochs - 1

cp = model_checkpoints[visualize_epoch]

batch_inputs, _ = next(iter(data_loader))

idx = torch.randperm(batch_inputs.shape[0])[:25]
inputs_25 = batch_inputs[idx].to(device)

for m in [cp["normal"], cp["sparse"], cp["wta_fc"], cp["wta_conv"]]:
    m.eval()

with torch.no_grad():
    recon_normal   = cp["normal"](inputs_25).cpu()
    recon_sparse   = cp["sparse"](inputs_25).cpu()
    recon_wta_fc   = cp["wta_fc"](inputs_25).cpu()
    recon_wta_conv = cp["wta_conv"](inputs_25).cpu()

try:
    C, H, W = get_patch_shape(dataset)
except ValueError:
    s = int(math.sqrt(input_dim))
    C, H, W = 1, s, s

orig_vis         = inputs_25.cpu().reshape(-1, C, H, W)
rec_normal_vis   = recon_normal.reshape(-1, C, H, W)
rec_sparse_vis   = recon_sparse.reshape(-1, C, H, W)
rec_wta_fc_vis   = recon_wta_fc.reshape(-1, C, H, W)
rec_wta_conv_vis = recon_wta_conv.reshape(-1, C, H, W)

# Build grids
grid_orig     = make_grid(orig_vis,         nrow=5, normalize=True, scale_each=True, padding=2)
grid_normal   = make_grid(rec_normal_vis,   nrow=5, normalize=True, scale_each=True, padding=2)
grid_sparse   = make_grid(rec_sparse_vis,   nrow=5, normalize=True, scale_each=True, padding=2)
grid_wta_fc   = make_grid(rec_wta_fc_vis,   nrow=5, normalize=True, scale_each=True, padding=2)
grid_wta_conv = make_grid(rec_wta_conv_vis, nrow=5, normalize=True, scale_each=True, padding=2)

fig, axes = plt.subplots(5, 1, figsize=(8, 18))

axes[0].imshow(grid_orig.permute(1, 2, 0).clamp(0, 1))
axes[0].axis("off")
axes[0].set_title("Original patches (25)")

axes[1].imshow(grid_normal.permute(1, 2, 0).clamp(0, 1))
axes[1].axis("off")
axes[1].set_title("Autoencoder reconstructions")

axes[2].imshow(grid_sparse.permute(1, 2, 0).clamp(0, 1))
axes[2].axis("off")
axes[2].set_title("K-Sparse AE reconstructions")

axes[3].imshow(grid_wta_fc.permute(1, 2, 0).clamp(0, 1))
axes[3].axis("off")
axes[3].set_title("WTA-FC AE reconstructions")

axes[4].imshow(grid_wta_conv.permute(1, 2, 0).clamp(0, 1))
axes[4].axis("off")
axes[4].set_title("WTA-CONV AE reconstructions")

plt.tight_layout()
plt.show()

#### 7. Angles

In [ ]:
NUM_ANGLE_SAMPLES = 500
POOL_SIZE = 2000

visualize_epoch = epochs - 1
cp = model_checkpoints[visualize_epoch]

def cosine_sim(a: torch.Tensor, b: torch.Tensor) -> float:
    a_flat = a.flatten().float()
    b_flat = b.flatten().float()
    return F.cosine_similarity(a_flat.unsqueeze(0), b_flat.unsqueeze(0)).item()

normal_model   = cp["normal"].to(device).eval()
sparse_model   = cp["sparse"].to(device).eval()
wta_fc_model   = cp["wta_fc"].to(device).eval()
wta_conv_model = cp["wta_conv"].to(device).eval()

# resnet_model = AutoModelForImageClassification.from_pretrained(
#     "edadaltocg/resnet18_cifar10", ignore_mismatched_sizes=True
# ).to(device).eval()
# _resnet_mean = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1)
# _resnet_std  = torch.tensor([0.2023, 0.1994, 0.2010], device=device).view(1, 3, 1, 1)

data_pool = []
for batch, _ in data_loader:
    data_pool.append(batch)
    if sum(b.shape[0] for b in data_pool) >= POOL_SIZE:
        break
data_pool = torch.cat(data_pool, dim=0)[:POOL_SIZE]

angles_before   = []
angles_normal   = []
angles_sparse   = []
angles_wta_fc   = []
angles_wta_conv = []
# angles_resnet   = []

for _ in tqdm(range(NUM_ANGLE_SAMPLES), desc="Angle samples"):
    i, j = torch.randperm(POOL_SIZE)[:2].tolist()
    x1 = data_pool[i : i + 1].to(device)
    x2 = data_pool[j : j + 1].to(device)
    angles_before.append(cosine_sim(x1, x2))

    with torch.no_grad():
        normal_model(x1); l1_normal = normal_model.last_latent
        normal_model(x2); l2_normal = normal_model.last_latent
        angles_normal.append(cosine_sim(l1_normal, l2_normal))

        sparse_model(x1); l1_sparse = sparse_model.last_latent
        sparse_model(x2); l2_sparse = sparse_model.last_latent
        angles_sparse.append(cosine_sim(l1_sparse, l2_sparse))

        wta_fc_model(x1); l1_wta_fc = wta_fc_model.last_latent
        wta_fc_model(x2); l2_wta_fc = wta_fc_model.last_latent
        angles_wta_fc.append(cosine_sim(l1_wta_fc, l2_wta_fc))

        wta_conv_model(x1); l1_wta_conv = wta_conv_model.last_latent
        wta_conv_model(x2); l2_wta_conv = wta_conv_model.last_latent
        angles_wta_conv.append(cosine_sim(l1_wta_conv, l2_wta_conv))

        # x1_resnet = F.interpolate(x1.view(1, C, H, W), size=(32, 32), mode="bilinear", align_corners=False)
        # x2_resnet = F.interpolate(x2.view(1, C, H, W), size=(32, 32), mode="bilinear", align_corners=False)
        # x1_resnet = (x1_resnet - _resnet_mean) / _resnet_std
        # x2_resnet = (x2_resnet - _resnet_mean) / _resnet_std
        # _conv1 = resnet_model.timm_model.conv1
        # feat1 = _conv1(x1_resnet)
        # feat2 = _conv1(x2_resnet)
        # angles_resnet.append(cosine_sim(feat1, feat2))

all_models       = ["Autoencoder", "K-Sparse AE", "WTA FC AE",   "WTA Conv AE",   "ResNet18"]
all_angles_after = [angles_normal, angles_sparse, angles_wta_fc, angles_wta_conv, angles_resnet]

n_models = len(all_models)
n_cols = 3
n_rows = math.ceil(n_models / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 5.0 * n_rows))
axes = axes.flatten()

for ax, title, angles_after in zip(axes, all_models, all_angles_after):
    ax.scatter(angles_before, angles_after, alpha=0.5, s=8)
    all_vals = angles_before + angles_after
    lo, hi = min(all_vals), max(all_vals)
    pad = (hi - lo) * 0.05
    lo, hi = lo - pad, hi + pad
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.5)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel("Cosine sim before")
    ax.set_ylabel("Cosine sim after")
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

for ax in axes[n_models:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()